all files in adventureWorks2 with column names

In [7]:
from notebookutils import mssparkutils
from pyspark.sql import functions as F

account   = "onelake"
container = "Fabmigration"

bronze_base = f"abfss://{container}@{account}.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks2/"
silver_base = f"abfss://{container}@{account}.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/"

print("BRONZE:", bronze_base)
print("SILVER:", silver_base)


StatementMeta(, f3fd4f15-87e9-4d30-a352-42b6c1afdf58, 9, Finished, Available, Finished, False)

BRONZE: abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks2/
SILVER: abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/


In [8]:
files = mssparkutils.fs.ls(bronze_base)

production_files = [
    f"{bronze_base}{f.name}"
    for f in files
    if f.name.startswith("Production.") and f.name.endswith(".parquet")
]


print("Production.*  files found:")
for p in production_files:
    print(" -", p)


StatementMeta(, f3fd4f15-87e9-4d30-a352-42b6c1afdf58, 10, Finished, Available, Finished, False)

Production.*  files found:
 - abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks2/Production.BillOfMaterials.parquet
 - abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks2/Production.Culture.parquet
 - abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks2/Production.Illustration.parquet
 - abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks2/Production.Location.parquet
 - abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks2/Production.ProductCategory.parquet
 - abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks2/Production.ProductCostHistory.parquet
 - abfss://Fabmi

In [9]:
def clean_df(df):
    # trim() a strings
    for col_name, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(col_name, F.trim(F.col(col_name)))

    # Replace nulls
    fill_dict = {col: "" for col, dtype in df.dtypes if dtype == "string"}
    fill_dict_numeric = {col: 0 for col, dtype in df.dtypes if dtype in ["int", "bigint", "double"]}
    df = df.fillna(fill_dict).fillna(fill_dict_numeric)

    return df


StatementMeta(, f3fd4f15-87e9-4d30-a352-42b6c1afdf58, 11, Finished, Available, Finished, False)

In [10]:
from typing import List, Optional
from pyspark.sql import DataFrame, functions as F, types as T

def standardizedate_df(
    df: DataFrame,
    date_cols: Optional[List[str]] = None,
    input_formats: Optional[List[str]] = None,
    cast_dates_as: str = "date",     # "date" or "timestamp"
    ingest_date: Optional[str] = None  # e.g., "2026-02-03" (casts to date); if None -> current_date()
) -> DataFrame:
    """
    1) Standardizes the provided or auto-detected date/timestamp columns:
       - Parses common string formats to timestamp
       - Casts to `date` (default) or keeps as `timestamp` via `cast_dates_as`
    2) Adds `ingest_date` column (date) for downstream partitioning.

    Parameters
    ----------
    df : input DataFrame
    date_cols : list of column names to treat as dates/timestamps. If None, auto-detects by name/ctype.
    input_formats : list of strptime-style formats to try in order
    cast_dates_as : "date" or "timestamp"
    ingest_date : string literal yyyy-MM-dd; if None uses current_date()
    """

    # --- 0) Trim strings
    for col_name, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(col_name, F.trim(F.col(col_name)))

    # --- 1) Which columns are date-like?
    if date_cols is None:
        # Heuristic: names that look like dates & are string/date/timestamp
        keywords = ("date", "dt", "_at", "_on", "timestamp", "ts", "fecha", "created", "updated")
        date_cols = [
            c for c, t in df.dtypes
            if t in ("string", "date", "timestamp") and any(k in c.lower() for k in keywords)
        ]

    # --- 2) Parse formats for string dates
    if input_formats is None:
        input_formats = [
            # Date only
            "yyyy-MM-dd", "yyyy/MM/dd", "dd-MM-yyyy", "dd/MM/yyyy", "MM-dd-yyyy", "MM/dd/yyyy",
            "yyyyMMdd", "ddMMyyyy",
            # Date + time
            "yyyy-MM-dd HH:mm:ss", "dd/MM/yyyy HH:mm:ss", "MM/dd/yyyy HH:mm:ss",
            # ISO-ish / with millis / TZ
            "yyyy-MM-dd'T'HH:mm:ss", "yyyy-MM-dd'T'HH:mm:ss.SSS",
            "yyyy-MM-dd'T'HH:mm:ssXXX", "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"
        ]

    dtype_map = dict(df.dtypes)
    for c in date_cols:
        t = dtype_map.get(c, "")

        # If it's already timestamp/date, just unify type later
        if t == "string":
            # Try multiple formats with coalesce
            parsed = None
            for fmt in input_formats:
                candidate = F.to_timestamp(F.col(c), fmt)
                parsed = candidate if parsed is None else F.coalesce(parsed, candidate)

            # Epoch seconds / millis (string or numeric) fallbacks
            parsed = F.coalesce(
                parsed,
                F.to_timestamp(F.col(c).cast("double")),                    # epoch seconds
                F.to_timestamp((F.col(c).cast("double")/1000.0))            # epoch millis
            )

            # If nothing matched, keep original to avoid data loss
            df = df.withColumn(c, F.when(parsed.isNotNull(), parsed).otherwise(F.col(c)))

        # Finally, cast to the requested final type
        if cast_dates_as == "date":
            df = df.withColumn(c, F.to_date(F.col(c)))
        else:
            df = df.withColumn(c, F.to_timestamp(F.col(c)))

    # --- 3) Add ingest_date for partitioning
    if ingest_date:
        df = df.withColumn("ingest_date", F.lit(ingest_date).cast(T.DateType()))
    else:
        df = df.withColumn("ingest_date", F.current_date())

    return df

StatementMeta(, f3fd4f15-87e9-4d30-a352-42b6c1afdf58, 12, Finished, Available, Finished, False)

In [11]:
for file_path in production_files:
    print(f"\nProcessing: {file_path}")

    # Base name, ej: Production.Culture.parquet
    #file_name = file_path.split("/")[-1].replace(".parquet", "")
    #display(file_name)

    # Base name
    file_name = file_path.split("/")[-1]             
    file_name = file_name.replace(".parquet", "") 
    file_name = file_name.replace(".", "_")   
    file_name = "silver_" + file_name.lower()         
    display(file_name)

    # Read parquet file
    df = spark.read.parquet(file_path) 
    df.printSchema()

    # Clean
    df_clean = clean_df(df)

    # dates
    df_table= standardizedate_df(df_clean)

    # Destination silver path
    out_path = f"{silver_base}{file_name}/"   
    print(f"Saved in Silver: {out_path}")

    # Save  (Silver)  
    df_table.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(out_path)    

    #Next lets register the delta table to use in Synapse
    # Create schema if not exist
    targetEnv = "silver"
    spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {targetEnv}""")
    
    #Register table to use in Synapse
    spark.sql(f""" CREATE TABLE IF NOT EXISTS {targetEnv}.{file_name.replace("silver_", "").lower()} USING DELTA LOCATION '{out_path}/' """)
   

StatementMeta(, f3fd4f15-87e9-4d30-a352-42b6c1afdf58, 13, Finished, Available, Finished, False)


Processing: abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks2/Production.BillOfMaterials.parquet


'silver_production_billofmaterials'

root
 |-- BillOfMaterialsID: integer (nullable = true)
 |-- ProductAssemblyID: integer (nullable = true)
 |-- ComponentID: integer (nullable = true)
 |-- StartDate: timestamp (nullable = true)
 |-- EndDate: timestamp (nullable = true)
 |-- UnitMeasureCode: string (nullable = true)
 |-- BOMLevel: integer (nullable = true)
 |-- PerAssemblyQty: decimal(8,2) (nullable = true)
 |-- ModifiedDate: timestamp (nullable = true)

Saved in Silver: abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/silver_production_billofmaterials/

Processing: abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks2/Production.Culture.parquet


'silver_production_culture'

...

root
 |-- UnitMeasureCode: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- ModifiedDate: timestamp (nullable = true)

Saved in Silver: abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/silver_production_unitmeasure/

Processing: abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks2/Production.WorkOrder.parquet


'silver_production_workorder'

root
 |-- WorkOrderID: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- OrderQty: integer (nullable = true)
 |-- StockedQty: integer (nullable = true)
 |-- ScrappedQty: integer (nullable = true)
 |-- StartDate: timestamp (nullable = true)
 |-- EndDate: timestamp (nullable = true)
 |-- DueDate: timestamp (nullable = true)
 |-- ScrapReasonID: integer (nullable = true)
 |-- ModifiedDate: timestamp (nullable = true)

Saved in Silver: abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/silver_production_workorder/

Processing: abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks2/Production.WorkOrderRouting.parquet


'silver_production_workorderrouting'

root
 |-- WorkOrderID: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- OperationSequence: integer (nullable = true)
 |-- LocationID: integer (nullable = true)
 |-- ScheduledStartDate: timestamp (nullable = true)
 |-- ScheduledEndDate: timestamp (nullable = true)
 |-- ActualStartDate: timestamp (nullable = true)
 |-- ActualEndDate: timestamp (nullable = true)
 |-- ActualResourceHrs: decimal(9,4) (nullable = true)
 |-- PlannedCost: decimal(19,4) (nullable = true)
 |-- ActualCost: decimal(19,4) (nullable = true)
 |-- ModifiedDate: timestamp (nullable = true)

Saved in Silver: abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/silver_production_workorderrouting/


In [41]:
"""
# Base path 
base_path = "abfss://fabmigration@fabmigration.dfs.core.windows.net/silver/adventureworks/"

# Synapse util
from notebookutils import mssparkutils

dry_run = True   # ← Change to False to remove

items = mssparkutils.fs.ls(base_path)
targets = [it for it in items if it.isDir and it.name.strip("/").lower().startswith("production_")]

print("Folders founded starting with 'production_':")
for t in targets:
    print(" -", t.path)

if not dry_run:
    for t in targets:
        print("Borrando →", t.path)
        mssparkutils.fs.rm(t.path, True)   # True = recursive
    print("Done ✔")
else:
    print("\nDry-run active: nothing removed. Change dry_run=False to execute.")
   """

In [ ]:
#Create external tables
#for file_path in production_files:
#    table_name = file_path.split("/")[-1].replace(".parquet", "")
#    out_path = f"{silver_base}{table_name}/"
#
#    spark.sql(f"""
#        CREATE OR REPLACE TABLE silver_{table_name}
#        USING PARQUET
#        LOCATION '{out_path}'
#    """)

In [10]:
from notebookutils import mssparkutils
from pyspark.sql import functions as F

account   = "onelake"
container = "Fabmigration"

bronze_base = f"abfss://{container}@{account}.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks/"
silver_base = f"abfss://{container}@{account}.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/"

print("BRONZE:", bronze_base)
print("SILVER:", silver_base)

StatementMeta(, 24d15aec-f6b6-4751-8ebc-c8575b1ace8d, 12, Finished, Available, Finished, False)

BRONZE: abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks/
SILVER: abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/


In [12]:
files = mssparkutils.fs.ls(bronze_base)

production_files = [
    f"{bronze_base}{f.name}"
    for f in files
    if f.name.startswith("Person.") and f.name.endswith(".parquet")
]


print("person.*  files found:")
for p in production_files:
    print(" -", p)

StatementMeta(, 24d15aec-f6b6-4751-8ebc-c8575b1ace8d, 14, Finished, Available, Finished, False)

person.*  files found:
 - abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks/Person.Address.parquet
 - abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks/Person.AddressType.parquet
 - abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks/Person.BusinessEntity.parquet
 - abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks/Person.BusinessEntityAddress.parquet
 - abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks/Person.BusinessEntityContact.parquet
 - abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks/Person.ContactType.parquet
 - abfss://Fabmigration@onelake.dfs.fabr

In [13]:
for file_path in production_files:
    print(f"\nProcessing: {file_path}")

    # Base name, ej: Production.Culture.parquet
    #file_name = file_path.split("/")[-1].replace(".parquet", "")
    #display(file_name)

    # Base name
    file_name = file_path.split("/")[-1]             
    file_name = file_name.replace(".parquet", "") 
    file_name = file_name.replace(".", "_")   
    file_name = "silver_" + file_name.lower()         
    display(file_name)

    # Destination silver path
    out_path = f"{silver_base}{file_name}/"   
    print(f"Saved in Silver: {out_path}")



    #Next lets register the delta table to use in Synapse
    # Create schema if not exist
    targetEnv = "silver"
    #spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {targetEnv}""")
    
    #Register table to use in Synapse
    spark.sql(f""" CREATE TABLE IF NOT EXISTS {targetEnv}.{file_name.replace("silver_", "").lower()} USING DELTA LOCATION '{out_path}/' """)
   

StatementMeta(, 24d15aec-f6b6-4751-8ebc-c8575b1ace8d, 15, Finished, Available, Finished, False)


Processing: abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks/Person.Address.parquet


'silver_person_address'

Saved in Silver: abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/silver_person_address/


AnalysisException: [DELTA_CREATE_EXTERNAL_TABLE_WITHOUT_TXN_LOG] 
You are trying to create an external table `chimcobldhq2ahj1c9mmipric5q6irre4l9nirj1e1pmarb9ctp62t39dtn2asr9dhr6asg`.`person_address`
from `abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/silver_person_address` using Delta, but there is no transaction log present at
`abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/silver_person_address/_delta_log`. Check the upstream job to make sure that it is writing using
format("delta") and that the path is the root of the table.

To learn more about Delta, see abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/silver_person_address